# Tabular Anomaly Detection: Quantum vs Classical Methods

This notebook demonstrates multiple anomaly detection approaches on tabular data,
comparing **quantum** methods (kernel One-Class SVM, variational autoencoder, QAOA clustering,
quantum distance k-NN) against **classical** baselines (Isolation Forest, One-Class SVM, LOF,
Elliptic Envelope, simulated annealing).

We use synthetic blob data as the primary dataset for fast, reproducible experiments.
The credit card fraud dataset is also available for more realistic evaluation.

In [ ]:
%matplotlib inline

# === Configuration (adjust these as needed) ===
SEED = 42
N_QUBITS = 6            # number of qubits / PCA components
N_SAMPLES = 2000         # total samples in synthetic dataset
CONTAMINATION = 0.05     # expected fraction of anomalies
N_KERNEL_SAMPLES = 150   # subset size for quantum kernel (slow)
N_QAOA_POINTS = 12       # subset size for QAOA clustering (very slow)
CIRCUIT_SCALE = 0.6  # Scale factor for circuit drawings


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist, squareform

## 1. Data Loading & Exploratory Analysis

We use `load_synthetic_blobs` to generate a controlled dataset with a known anomaly
fraction. This is faster and more reproducible than the full credit card dataset.

> **Tip:** For real-world evaluation, swap in `load_creditcard(subsample=5000, seed=42)`.

In [ ]:
from quantum_anomaly_detection.data.tabular import (
    load_synthetic_blobs, load_creditcard, preprocess_tabular
)

X_raw, y = load_synthetic_blobs(
    n_samples=N_SAMPLES, n_features=N_QUBITS,
    anomaly_fraction=CONTAMINATION, seed=SEED
)

print(f"Dataset shape: {X_raw.shape}")
print(f"Class distribution: normal={np.sum(y == 0)}, anomaly={np.sum(y == 1)}")
print(f"Anomaly fraction: {np.mean(y):.2%}")

In [ ]:
# Scatter plot of the first two features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, name in [(0, "steelblue", "Normal"), (1, "crimson", "Anomaly")]:
    mask = y == label
    axes[0].scatter(X_raw[mask, 0], X_raw[mask, 1], c=color, label=name, alpha=0.5, s=10)
axes[0].set_xlabel("Feature 0")
axes[0].set_ylabel("Feature 1")
axes[0].set_title("Raw features (first two dimensions)")
axes[0].legend()

axes[1].bar(["Normal", "Anomaly"], [np.sum(y == 0), np.sum(y == 1)],
            color=["steelblue", "crimson"])
axes[1].set_title("Class distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 2. Preprocessing & Train/Test Split

We split the data 80/20, then apply PCA (to `N_QUBITS` components) and scale
features into $[0, \pi]$ for angle encoding on quantum circuits.

The `preprocess_tabular` function is fitted on training data and then applied
to the test set to avoid data leakage.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, stratify=y
)

# Fit preprocessing on training data, apply to both splits
X_train = preprocess_tabular(X_train_raw, n_components=N_QUBITS)
X_test = preprocess_tabular(X_test_raw, n_components=N_QUBITS, fit_data=X_train_raw)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train labels: normal={np.sum(y_train == 0)}, anomaly={np.sum(y_train == 1)}")
print(f"Test labels:  normal={np.sum(y_test == 0)}, anomaly={np.sum(y_test == 1)}")

# Prepare normal-only training set (for one-class methods)
X_train_normal = X_train[y_train == 0]

---
## 3. Quantum Kernel + One-Class SVM

Quantum kernel methods map data into exponentially large Hilbert spaces using
parameterized quantum circuits. The **ZZ feature map** encodes classical features
as rotation angles and introduces entanglement via $ZZ$ interactions between
neighbouring qubits, creating a kernel that captures correlations classical
RBF kernels may miss.

We estimate the kernel matrix $K_{ij} = |\langle \phi(x_i) | \phi(x_j) \rangle|^2$
via statevector simulation and feed it into a one-class SVM. The SVM finds a
boundary in feature space that encloses most of the training (normal) data; points
outside the boundary are flagged as anomalies.

Because kernel matrix computation scales as $O(n^2)$ circuit evaluations, we use a
small subset of the data (`N_KERNEL_SAMPLES`).

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_zz_feature_map, build_angle_encoding_map
from quantum_anomaly_detection.methods.quantum_kernel import quantum_kernel_svm, compute_kernel_matrix

# ZZ feature map (reps=1: shallower circuit, less overfitting)
zz_fmap = build_zz_feature_map(N_QUBITS, reps=1)
print('ZZ Feature Map:')
display(zz_fmap.draw('mpl', fold=20, scale=CIRCUIT_SCALE))

# Angle encoding feature map (hand-built)
angle_fmap = build_angle_encoding_map(N_QUBITS)
print('\nAngle Encoding Feature Map:')
display(angle_fmap.draw('mpl', fold=20, scale=CIRCUIT_SCALE))


In [ ]:
# Subsample for kernel computation (O(n^2) circuit evaluations)
rng = np.random.default_rng(SEED)
n_train_kernel = min(N_KERNEL_SAMPLES, len(X_train_normal))
n_test_kernel = min(N_KERNEL_SAMPLES, len(X_test))
idx_train_k = rng.choice(len(X_train_normal), n_train_kernel, replace=False)
idx_test_k = rng.choice(len(X_test), n_test_kernel, replace=False)

X_train_k = X_train_normal[idx_train_k]
X_test_k = X_test[idx_test_k]
y_test_k = y_test[idx_test_k]

print(f"Kernel subset: {n_train_kernel} train, {n_test_kernel} test")

In [ ]:
# Compute and visualise the quantum kernel matrix on training data
from quantum_anomaly_detection.visualization.plots import plot_kernel_matrix

K_train = compute_kernel_matrix(X_train_k, zz_fmap)
plot_kernel_matrix(K_train, title="Quantum Kernel Matrix (train subset)")
plt.show()

In [ ]:
# Compare quantum kernel with both feature maps
preds_zz, scores_zz, _ = quantum_kernel_svm(
    X_train_k, X_test_k, zz_fmap, nu=0.1
)
preds_angle, scores_angle, _ = quantum_kernel_svm(
    X_train_k, X_test_k, angle_fmap, nu=0.1
)

from quantum_anomaly_detection.evaluation.metrics import compute_metrics

m_zz = compute_metrics(y_test_k, preds_zz, scores_zz)
m_angle = compute_metrics(y_test_k, preds_angle, scores_angle)

print('Quantum Kernel SVM — Feature Map Comparison:')
print(f'  ZZ (reps=1):      {" | ".join(f"{k}={v:.3f}" for k, v in m_zz.items())}')
print(f'  Angle encoding:   {" | ".join(f"{k}={v:.3f}" for k, v in m_angle.items())}')
print()
print('Note: quantum kernel accuracy depends heavily on feature map choice,\n'
      'PCA components, and sample size. Increasing N_KERNEL_SAMPLES improves\n'
      'results but costs O(n^2) kernel computation.')


---
## 4. Variational Quantum Autoencoder

A variational quantum autoencoder (VQC-AE) compresses quantum states through a
bottleneck of `n_latent` qubits, then attempts to reconstruct the original state.
The circuit is trained on **normal** data so that it learns the structure of the
normal distribution. Anomalous inputs, which lie outside this learned manifold,
suffer higher reconstruction error.

The encoder and decoder are parameterized circuits optimized via classical
optimization (COBYLA). After training, the reconstruction loss for each test
sample serves as an anomaly score: higher loss indicates a likely anomaly.

We train on a small subset of normal data for speed; in practice, larger training
sets improve the learned representation.

In [ ]:
from quantum_anomaly_detection.circuits.autoencoder import build_autoencoder_compact

n_latent = N_QUBITS // 2
ae_compact, ae_encoder, ae_decoder = build_autoencoder_compact(
    N_QUBITS, n_latent, encoder_reps=1, decoder_reps=1
)
print(f'Autoencoder: {N_QUBITS} qubits, {n_latent} latent')
print('\nCompact view:')
display(ae_compact.draw('mpl', scale=CIRCUIT_SCALE))
print('\nEncoder detail:')
display(ae_encoder.draw('mpl', fold=20, scale=CIRCUIT_SCALE))
print('\nDecoder detail:')
display(ae_decoder.draw('mpl', fold=20, scale=CIRCUIT_SCALE))


In [ ]:
# Train on a small subset of normal data (autoencoder training is slow)
n_ae_train = 50
X_ae_train = X_train_normal[:n_ae_train]

print(f"Training autoencoder on {n_ae_train} normal samples...")
ae_params, ae_circuit, ae_cost_history = train_autoencoder(
    X_ae_train, n_latent,
    encoder_reps=2, decoder_reps=2,
    maxiter=200, seed=SEED, method="COBYLA"
)

plot_optimization_history(ae_cost_history, title="Autoencoder Training")
plt.show()

In [ ]:
# Score and detect anomalies on the test set
ae_scores = score_anomalies(X_test, ae_params, ae_circuit, n_latent)
ae_preds = detect_anomalies(ae_scores, contamination=CONTAMINATION)

ae_metrics = compute_metrics(y_test, ae_preds, scores=ae_scores)
print("VQC Autoencoder:")
for k, v in ae_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_anomaly_scores

plot_anomaly_scores(ae_scores, y_test, title="Autoencoder Reconstruction Loss")
plt.show()

---
## 5. QAOA Clustering

The Quantum Approximate Optimization Algorithm (QAOA) can solve combinatorial
optimization problems on near-term quantum hardware. Here we formulate anomaly
detection as a graph-based clustering problem: given pairwise distances between
data points, find a partition that minimizes intra-cluster distances.

Small, isolated clusters in the resulting partition are then flagged as anomalies.
This approach is particularly interesting because it frames anomaly detection as
a discrete optimization problem natively suited to quantum computation.

QAOA scales exponentially with the number of data points (one qubit per point),
so we restrict this experiment to a very small subset (`N_QAOA_POINTS`).

In [ ]:
from quantum_anomaly_detection.methods.qaoa_clustering import (
    run_qaoa_clustering, identify_anomaly_clusters
)

# Subsample for QAOA (one qubit per data point)
# Ensure subsample includes some anomalies
idx_anomaly_test = np.where(y_test == 1)[0]
idx_normal_test = np.where(y_test == 0)[0]
n_anom = min(3, len(idx_anomaly_test))
n_norm = N_QAOA_POINTS - n_anom
idx_qaoa = np.concatenate([
    rng.choice(idx_normal_test, n_norm, replace=False),
    rng.choice(idx_anomaly_test, n_anom, replace=False),
])
X_qaoa = X_test[idx_qaoa]
y_qaoa = y_test[idx_qaoa]

print(f"QAOA subset: {N_QAOA_POINTS} points "
      f"(normal={np.sum(y_qaoa == 0)}, anomaly={np.sum(y_qaoa == 1)})")

In [ ]:
print("Running QAOA clustering...")
qaoa_labels, qaoa_cost_history = run_qaoa_clustering(
    X_qaoa, reps=2, maxiter=200, seed=SEED
)

plot_optimization_history(qaoa_cost_history, title="QAOA Clustering")
plt.show()

In [ ]:
# Identify small clusters as anomalies
qaoa_preds = identify_anomaly_clusters(
    qaoa_labels, X_qaoa, min_cluster_fraction=0.1
)

qaoa_metrics = compute_metrics(y_qaoa, qaoa_preds)
print("QAOA Clustering:")
for k, v in qaoa_metrics.items():
    print(f"  {k}: {v:.4f}")

---
## 6. Quantum Distance k-NN

This method uses the **swap test** circuit to estimate the fidelity (overlap)
between quantum-encoded data points, converting fidelity into a distance metric.
The resulting distance matrix drives a k-nearest-neighbour anomaly scorer:
points whose k nearest neighbours are far away receive high anomaly scores.

The swap test computes $|\langle \psi | \phi \rangle|^2$ with a single ancilla
qubit, making it a natural primitive for quantum-enhanced distance computation.
However, computing the full $O(n^2)$ distance matrix is expensive, so we again
work on a subset of the data.

In [ ]:
from quantum_anomaly_detection.methods.quantum_distance import detect_anomalies_knn
from quantum_anomaly_detection.circuits.feature_maps import build_angle_encoding_map

# Build feature map for distance computation
angle_fmap = build_angle_encoding_map(N_QUBITS)

# Subsample for distance computation (~80 points)
n_dist = 80
idx_dist = rng.choice(len(X_test), n_dist, replace=False)
X_dist = X_test[idx_dist]
y_dist = y_test[idx_dist]

print(f"Distance k-NN subset: {n_dist} points")
knn_preds, knn_scores = detect_anomalies_knn(
    X_dist, angle_fmap, k=5, contamination=CONTAMINATION
)

knn_metrics = compute_metrics(y_dist, knn_preds, scores=knn_scores)
print("Quantum Distance k-NN:")
for k, v in knn_metrics.items():
    print(f"  {k}: {v:.4f}")

---
## 7. Classical Benchmarks

### Level A: Standard methods on the full test set

In [ ]:
from quantum_anomaly_detection.classical.benchmarks import (
    run_isolation_forest, run_svm, run_lof,
    run_elliptic_envelope, run_simulated_annealing
)

# Isolation Forest
if_preds, if_scores = run_isolation_forest(
    X_train_normal, X_test, contamination=CONTAMINATION, seed=SEED
)
if_metrics = compute_metrics(y_test, if_preds, scores=if_scores)

# One-Class SVM (classical RBF kernel)
ocsvm_preds, ocsvm_scores = run_svm(
    X_train_normal, X_test, kernel="rbf", nu=CONTAMINATION
)
ocsvm_metrics = compute_metrics(y_test, ocsvm_preds, scores=ocsvm_scores)

# Local Outlier Factor
lof_preds, lof_scores = run_lof(
    X_train_normal, X_test, n_neighbors=20, contamination=CONTAMINATION
)
lof_metrics = compute_metrics(y_test, lof_preds, scores=lof_scores)

# Elliptic Envelope
ee_preds, ee_scores = run_elliptic_envelope(
    X_train_normal, X_test, contamination=CONTAMINATION
)
ee_metrics = compute_metrics(y_test, ee_preds, scores=ee_scores)

print("Classical benchmarks computed on full test set.")

### Level B: Simulated Annealing on the QAOA subsample

To compare QAOA fairly, we run simulated annealing on the same small subset
using a classical pairwise distance matrix as the cost.

In [ ]:
# Build cost matrix from pairwise Euclidean distances
cost_matrix = squareform(pdist(X_qaoa))

sa_labels, sa_cost_history = run_simulated_annealing(
    cost_matrix, n_iter=10000, seed=SEED
)

# Use the same anomaly-cluster identification
sa_preds = identify_anomaly_clusters(
    sa_labels, X_qaoa, min_cluster_fraction=0.1
)
sa_metrics = compute_metrics(y_qaoa, sa_preds)

print("Simulated Annealing (QAOA subset):")
for k, v in sa_metrics.items():
    print(f"  {k}: {v:.4f}")

---
## 8. Comparison & Discussion

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import format_results_table

# Collect all results
results = {
    "Quantum Kernel SVM": qk_metrics,
    "VQC Autoencoder": ae_metrics,
    "QAOA Clustering": qaoa_metrics,
    "Quantum k-NN": knn_metrics,
    "Isolation Forest": if_metrics,
    "Classical One-Class SVM": ocsvm_metrics,
    "LOF": lof_metrics,
    "Elliptic Envelope": ee_metrics,
    "Sim. Annealing (QAOA sub)": sa_metrics,
}

results_df = format_results_table(results)
results_df

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_roc_curves

# ROC curves for methods that have scores on the full test set
roc_data = {
    "VQC Autoencoder": (y_test, ae_scores),
    "Isolation Forest": (y_test, if_scores),
    "Classical One-Class SVM": (y_test, ocsvm_scores),
    "LOF": (y_test, lof_scores),
    "Elliptic Envelope": (y_test, ee_scores),
}

# Add kernel and k-NN on their respective subsets
roc_data["Quantum Kernel One-Class SVM (sub)"] = (y_test_k, qk_scores)
roc_data["Quantum k-NN (sub)"] = (y_dist, knn_scores)

plot_roc_curves(roc_data)
plt.show()

### Discussion

**Key observations:**

- **Classical methods** (Isolation Forest, One-Class SVM, LOF, Elliptic Envelope) run on
  the full test set and are fast. They serve as strong baselines.
- **Quantum Kernel One-Class SVM** can capture non-trivial feature interactions via the ZZ
  entangling map, but is limited by the $O(n^2)$ kernel matrix cost.
- **VQC Autoencoder** learns a compressed quantum representation of normality.
  Performance depends heavily on training data size and optimization convergence.
- **QAOA Clustering** is currently limited to very small subsets due to qubit
  requirements (one qubit per data point). It serves as a proof of concept for
  combinatorial anomaly detection.
- **Quantum Distance k-NN** provides a quantum-native distance metric via the
  swap test, but shares the $O(n^2)$ scaling challenge.

These quantum methods demonstrate conceptual advantages (exponential feature
spaces, native handling of quantum states), but practical quantum advantage
for tabular anomaly detection remains an open research question. Scaling to
larger datasets will require error-mitigated hardware or improved classical
simulation techniques.